# 06. Language Detection — Azure AI Language (`detect_language`)

This notebook calls the Azure AI Language service's **language detection** operation via
`TextAnalyticsClient.detect_language()`. It runs detection over four documents — English, French, Japanese,
and a two-letter ambiguous string (`"OK"`) — to show both confident detection and the service's behavior on
text that's too short/ambiguous to classify reliably.

**Difficulty:** Beginner


## Prerequisites

**Python packages:**
- `azure-ai-textanalytics` — **not yet in the repo's root `requirements.txt`**, install with:
  ```
  pip3 install azure-ai-textanalytics
  ```
- `python-dotenv` (already in `requirements.txt`)

**Azure resources required:**
- An Azure AI Language resource with a key and endpoint

**Environment variables** (add to root `.env`):
```
AZURE_LANGUAGE_ENDPOINT=https://<your-language-resource>.services.ai.azure.com/
AZURE_LANGUAGE_KEY=<your-language-resource-key>
```


## What You'll Learn

- How to call `detect_language()` on a batch of documents in different languages
- The shape of a `DetectedLanguage` result: `.name`, `.iso6391_name`, `.confidence_score`
- How the service handles genuinely ambiguous input (e.g. a bare `"OK"`) rather than forcing a guess
- Why language detection is typically the *first* step in a pipeline that later calls NER/PII/sentiment


### Step 1 — Create the Text Analytics client

Same client class and auth pattern as notebook 05 (`TextAnalyticsClient` + `AzureKeyCredential`) — language
detection is just another operation on the same client.

💡 **Exam tip:** All of `detect_language`, `recognize_entities`, `recognize_pii_entities`,
`analyze_sentiment`, and `extract_key_phrases` live on the same `TextAnalyticsClient`. For the exam, remember
this single client class and pattern the operation name, not a different client per feature.

🔄 **Alternatives:** N/A for client setup — see notebook 05 for the REST-vs-SDK alternative.


In [ ]:
import os
from dotenv import load_dotenv
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

load_dotenv()

endpoint = os.environ["AZURE_LANGUAGE_ENDPOINT"]
api_key = os.environ["AZURE_LANGUAGE_KEY"]

client = TextAnalyticsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(api_key),
)

### Step 2 — Detect the language of several documents at once

Notice `detect_language()` is called **without** a `language=` argument, unlike `recognize_pii_entities` in
notebook 05 — that's the point of this operation: you don't know the language yet, so there's nothing to pass.
The four sample documents cover English, French, Japanese, and a short, language-ambiguous string.

💡 **Exam tip:** Very short or generic text (like `"OK"`) can come back with the primary language reported as
`"(Unknown)"` / `iso6391_name == "unknown"` and a low confidence score, rather than the service guessing —
know that language detection confidence scores exist precisely to let your application decide whether to trust
a detection or fall back to a default/prompt the user.

🔄 **Alternatives:** If you already know the language out-of-band (e.g. from user account locale settings),
skip detection entirely and pass the known language code straight into `recognize_entities`/
`recognize_pii_entities`/`analyze_sentiment` — saves an API call and avoids misdetection on short text.


In [ ]:
documents = [
    "Hi, this is Sarah Chen from Acme Logistics regarding ticket TKT-1042.",
    "Bonjour, je vous écris au sujet du ticket TKT-1042 concernant notre VPN.",
    "こんにちは、TKT-1042のチケットについてVPNの問題をご連絡しています。",
    "OK"
]

response = client.detect_language(documents)
results = [doc for doc in response if not doc.is_error]

### Step 3 — Print the detected language and confidence score per document

Each result exposes `.primary_language`, a `DetectedLanguage` object with `.name` (human-readable, e.g.
`"French"`), `.iso6391_name` (the two-letter code, e.g. `"fr"`), and `.confidence_score` (0.0–1.0).

💡 **Exam tip:** `.iso6391_name` is the value you'd typically feed into other Language operations that expect
a `language=` code — know that Azure AI Language standardizes on ISO 639-1 codes across its APIs (with a small
number of exceptions for languages/scripts without a two-letter code).

🔄 **Alternatives:** For text mixing multiple languages in one document, be aware `detect_language` reports a
single *primary* language for the whole document — it does not segment a mixed-language document into
per-language spans. If you need that level of granularity, you'd need to split the text yourself (e.g. by
sentence) and detect each segment separately.


In [ ]:
for idx, doc in enumerate(results):
    primary = doc.primary_language
    print(f"--- Document {idx + 1}: \"{documents[idx][:40]}...\" ---")
    print(f"  Detected language: {primary.name} ({primary.iso6391_name})")
    print(f"  Confidence score: {primary.confidence_score:.2f}")
    print()

## Summary

This notebook ran Azure AI Language's `detect_language` operation over English, French, and Japanese text, plus
a deliberately ambiguous two-letter string, and printed the detected language name, ISO code, and confidence
score for each. The ambiguous case demonstrates that the service reports low confidence (or "unknown") rather
than forcing a guess — a detail worth building defensive logic around in any pipeline that chains language
detection into other Language operations.


## Try It Yourself

1. **Easy:** Add a Spanish and a German sentence to the `documents` list and confirm the detected `iso6391_name`
   values (`"es"`, `"de"`).
2. **Intermediate:** Write a small pipeline function that calls `detect_language` first, then feeds the
   detected `iso6391_name` into `recognize_entities` (notebook 07) for the same document — so NER always runs
   with the correct language code instead of a hardcoded `"en"`.
3. **Advanced:** Construct a document that deliberately mixes two languages in different sentences (e.g. one
   English sentence, one French sentence, concatenated) and observe what `detect_language` reports as the
   primary language and confidence score — reason about why.
